# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zoye-J/FlyRank--MachineLearning/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [20]:
import os, sys, subprocess, json

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Zoye-J/FlyRank--MachineLearning.git"
REPO_DIR = "FlyRank--MachineLearning"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "duckdb", "pandas", "numpy", "matplotlib"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)
os.makedirs("docs/figures", exist_ok=True)
print("Working dir:", os.getcwd())

Working dir: /content/FlyRank--MachineLearning/FlyRank--MachineLearning/FlyRank--MachineLearning/FlyRank--MachineLearning/FlyRank--MachineLearning/FlyRank--MachineLearning


## 1. Question


**Research question.** For pages that already have search visibility, which ones should a content editor review first for CTR and intent-alignment problems?

**Unit of analysis.** One row = one content page ("content_hash_id"), aggregated over a 30-day prior window.

**Output.** A priority score per page, sorted descending, with a reason code and an action label for the top of the queue.
A content editor opens the top of the queue and
rewrites the title/meta of pages whose CTR sits below the median for their
position tier or flags the page for deeper intent review. A false positive costs 2–4 editor hours. A false negative costs another quarter of lost clicks on a page that already has impressions: a large, measurable loss. The asymmetry (misses expensive, false positives cheap) justifies a wide top-of-queue and a conservative action floor.

Visibility, CTR, position, and freshness interact. A
single hand-written AND-rule captures some of that interaction; a learned model could capture more if the signal is real. This project tests whether it is and finds that on this slice, the hand rule wins at the top of the queue.

## 2. Data


**Release.** FlyRank ML Internship warehouse
("FlyRank/internship-warehouse", build v20260703), Hugging Face gated release.
Public-safe: no client names, URLs, or raw queries appear in this repo.

**Tables used.**
- 'fact_content_daily_performance/month=2026-03/*.parquet' daily per-page performance for the prior window.
- 'fact_content_daily_performance/month=2026-04/*.parquet' and month=2026-05/*.parquet' the future window used only to compute the evaluation label.
- 'dim_content.parquet' static page attributes ('content_type', 'main_intent', 'word_count', 'content_updated_date')

**Date windows.**
- Prior (feature) window: **2026-03-01 to 2026-03-31**.
- Future (label) window: **2026-04-01 to 2026-05-31**.
- The final month (month=2026-06) is sealed and never touched.

**Excluded fields, and why.**
- 'trend_direction', 'trend_pct'  starter-CSV only; label sources.
- `is_declining_future', 'imp_mar', 'imp_apr_may'  label and its inputs.
- 'content_hash_id', 'client_hash_id'  pseudonyms; grouping/joining only.
- 'ga4_*' when 'ga4_data_available = FALSE'  zeros are fill, not measurement.
- 'is_published', 'is_deleted', 'provider_used', 'model_used' product-decision metadata.
- All rows from month=2026-06  sealed test month.

**Grain check.** One row of the fact table = one page-day
('report_date, client_hash_id, content_hash_id'). Verified with a 'GROUP BY ... HAVING COUNT(*) > 1' probe that returned zero rows.

In [21]:
from google.colab import userdata
import duckdb
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("CREATE SECRET hf (TYPE huggingface, PROVIDER credential_chain);")

FACT_MAR = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
FACT_APR = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet"
FACT_MAY = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-05/*.parquet"
DIM_CONTENT = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

grain_check = con.execute(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
FROM '{FACT_MAR}'
GROUP BY 1,2,3 HAVING COUNT(*) > 1 LIMIT 5
""").df()
print(f"Grain check rows (must be 0): {len(grain_check)}")

counts = con.execute(f"""
SELECT COUNT(*) total_rows,
       COUNT(DISTINCT report_date) n_days,
       COUNT(DISTINCT content_hash_id) n_content,
       MIN(report_date) min_date, MAX(report_date) max_date
FROM '{FACT_MAR}'
""").df()
print(counts.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check rows (must be 0): 0
 total_rows  n_days  n_content   min_date   max_date
    9841378      31     331437 2026-03-01 2026-03-31


## 3. Methodology


**Baseline (frozen).** A hand-written rule:
score = visible × low_ctr × impressions_30d

where 'visible = impressions_30d >= 500', and 'low_ctr = ctr_30d < tier median'.
Reason codes: 'visible_lowctr', 'stale_visible', 'stale_only', 'not_flagged'.
Action labels: 'review_ctr', 'review_staleness', 'review_intent', 'no_action'.

**Model.** Logistic Regression on five raw prior-window signals:
'impressions_30d, clicks_30d, ctr_30d, avg_position_30d, days_since_update'.
Secondary: Decision Tree (depth 3). Gradient Boosting / Random Forest were not tried, LR already loses to the baseline, so adding complexity would not be honest.

**Label.** 'is_declining_future = 1' when the April–May impression sum is below 80% of the March impression sum. Observed, not defined by a rule.

**Validation.** 'GroupShuffleSplit' by 'client_hash_id', 'test_size=0.2',
'random_state=42'. Zero client overlap between train and test verified.

**Leakage checks.**
1. Feature list and label sources are disjoint (audited programmatically).
2. Train-without test on 'clicks_30d' P@50 unchanged, coefficient magnitude and top-of-list behaviour disagree.
3. Deliberate leak: adding 'imp_apr_may' as a feature jumps P@50 from 0.26 to 1.00  the harness catches it, and the leak is deleted.

**Honest framing of the result.** Held-out Precision@50 = 0.26 for LR, versus 0.52 for the hand rule on the same split. The model does not beat the rule at the top of the queue on this slice. That is the finding.

In [22]:
QUEUE_Q = f"""
WITH perf AS (
  SELECT content_hash_id,
         ANY_VALUE(client_hash_id)         AS client_hash_id,
         SUM(gsc_impressions)              AS impressions_30d,
         SUM(gsc_clicks)                   AS clicks_30d,
         AVG(NULLIF(gsc_avg_position, 0))  AS avg_position_30d
  FROM '{FACT_MAR}' GROUP BY content_hash_id
),
future AS (
  SELECT content_hash_id, SUM(gsc_impressions) AS imp_apr_may
  FROM (
    SELECT content_hash_id, gsc_impressions FROM '{FACT_APR}'
    UNION ALL
    SELECT content_hash_id, gsc_impressions FROM '{FACT_MAY}'
  ) GROUP BY content_hash_id
)
SELECT p.content_hash_id, p.client_hash_id, p.impressions_30d, p.clicks_30d,
       p.avg_position_30d, d.content_type, d.main_intent,
       DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') AS days_since_update,
       f.imp_apr_may
FROM perf p
LEFT JOIN '{DIM_CONTENT}' d ON p.content_hash_id = d.content_hash_id
LEFT JOIN future f ON p.content_hash_id = f.content_hash_id
WHERE p.impressions_30d IS NOT NULL AND p.impressions_30d > 0
"""
df = con.execute(QUEUE_Q).df()

df["ctr_30d"] = np.where(df["impressions_30d"] > 0,
                         100.0 * df["clicks_30d"] / df["impressions_30d"], np.nan)
df["position_tier"] = pd.cut(df["avg_position_30d"],
    bins=[0, 3, 10, 20, 50, 1e9],
    labels=["top_3","page_1","page_2","page_3_5","deep"])
df["tier_median_ctr"] = df["position_tier"].map(
    df.groupby("position_tier", observed=True)["ctr_30d"].median())

df["stale"]   = ((df["days_since_update"] >= 180) & (df["days_since_update"] >= 0)).astype(int)
df["visible"] = (df["impressions_30d"] >= 500).astype(int)
df["low_ctr"] = (df["ctr_30d"] < df["tier_median_ctr"]).astype(int)

df["baseline_score"] = (df["visible"] * df["low_ctr"] * df["impressions_30d"]).fillna(0)

def reason_code(r):
    if r["stale"] and r["visible"] and r["low_ctr"]: return "stale_visible_lowctr"
    if r["stale"] and r["visible"]:                  return "stale_visible"
    if r["stale"]:                                   return "stale_only"
    if r["visible"] and r["low_ctr"]:                return "visible_lowctr"
    return "not_flagged"

def action_label(r):
    if r["reason_code"] == "stale_visible_lowctr": return "review_intent"
    if r["reason_code"] == "visible_lowctr":       return "review_ctr"
    if r["reason_code"] == "stale_visible":        return "review_staleness"
    return "no_action"

df["reason_code"]  = df.apply(reason_code, axis=1)
df["action_label"] = df.apply(action_label, axis=1)

df["is_declining_future"] = np.where(
    df["imp_apr_may"].isna() | (df["impressions_30d"] == 0), np.nan,
    (df["imp_apr_may"] < 0.8 * df["impressions_30d"]).astype(float))

df = df.sort_values(["baseline_score","impressions_30d"],
                    ascending=[False,False]).reset_index(drop=True)
df["rank"] = np.arange(1, len(df)+1)

print(f"Modeling frame: {len(df):,} rows")
print(f"Base rate (future decline): {df['is_declining_future'].mean():.3f}")
print(f"Action counts:")
print(df["action_label"].value_counts().to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling frame: 176,738 rows
Base rate (future decline): 0.283
Action counts:
action_label
no_action           175364
review_ctr            1368
review_staleness         6


## 4. Results (vs baseline)

## 4. Results (vs baseline)

Both baseline and models evaluated on the same grouped split, same test set, same metric.

| Model | P@20 | P@50 | P@100 | AUC |
|---|---|---|---|---|
| Random ranking | ≈ base rate | ≈ base rate | ≈ base rate | 0.50 |
| Week-4 hand rule (baseline) | **0.45** | **0.52** | **0.45** | 0.51 |
| Logistic Regression | 0.25 | 0.26 | 0.36 | 0.56 |
| Decision Tree (depth 3) | 0.35 | 0.18 | 0.28 | 0.56 |

**Test-set base rate: 0.186. Train base rate: 0.310.**

**Reading.** The hand rule wins at Precision@20 and Precision@50, the part of the ranking a human actually acts on. LR has a slightly better AUC, meaning it discriminates across the whole test set better, but its top-of-queue quality is worse because it ranks low-impression noise first. The Decision Tree collapses at P@50.

**Error analysis.** The top-50 false positives from LR have median 3 impressions, mean 5,659 i.e. dominated by zero-traffic pages. The 51–200 bucket declines at 47.3% (well above the base rate) the model has signal but ranks it too low.

In [23]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_hash_id"]))
train_df, test_df = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()
assert len(set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])) == 0

FEATURES = ["impressions_30d","clicks_30d","ctr_30d","avg_position_30d","days_since_update"]
def make_X(d):
    X = d[FEATURES].copy()
    X["avg_position_30d"] = X["avg_position_30d"].fillna(-1)
    X["days_since_update"] = X["days_since_update"].fillna(-1)
    X["ctr_30d"] = X["ctr_30d"].fillna(0)
    return X
X_train, X_test = make_X(train_df), make_X(test_df)
y_train, y_test = train_df["is_declining_future"].values, test_df["is_declining_future"].values

scaler = StandardScaler()
Xtr_s, Xte_s = scaler.fit_transform(X_train), scaler.transform(X_test)
lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42).fit(Xtr_s, y_train)
dt = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_train, y_train)

def precision_at_k(scores, labels, k):
    return float(np.asarray(labels)[np.argsort(-np.asarray(scores))[:k]].mean())

def row(name, s):
    return {"model": name,
            "P@20": round(precision_at_k(s, y_test, 20), 4),
            "P@50": round(precision_at_k(s, y_test, 50), 4),
            "P@100": round(precision_at_k(s, y_test, 100), 4),
            "AUC": round(roc_auc_score(y_test, s), 4)}

random_scores = np.random.RandomState(42).uniform(size=len(y_test))
table = pd.DataFrame([
    row("Random ranking", random_scores),
    row("Week-4 baseline", test_df["baseline_score"].values),
    row("Logistic Regression", lr.predict_proba(Xte_s)[:, 1]),
    row("Decision Tree (d=3)", dt.predict_proba(X_test)[:, 1]),
])
print(f"Test base rate: {y_test.mean():.3f}")
print(table.to_string(index=False))

Test base rate: 0.186
              model  P@20  P@50  P@100    AUC
     Random ranking  0.15  0.22   0.16 0.4976
    Week-4 baseline  0.45  0.52   0.45 0.5088
Logistic Regression  0.25  0.26   0.36 0.5605
Decision Tree (d=3)  0.25  0.20   0.15 0.5611


## 5. Limitations


- **This is decision-support, not causal.** The label is observed but the analysis is cross-sectional. No claim that "fixing CTR causes recovery" is supported.
- **One month, one portfolio.** March 2026 prior window, April–May 2026 label, 47 clients in the modeling frame. Results may not generalize.

- **Grouped by client, not by time.** Features come from one window and labels from a later window, so the temporal axis is already honest. The generalization axis tested here is "unseen clients," not "unseen future months."
- **The baseline wins at P@50.** LR does not beat the hand rule at the top of the queue on this slice. The model's AUC advantage (0.56 vs 0.51) does not translate to better top-of-queue quality.
- **Staleness could not be used as a filter.** "content_updated_date" behaves like a sync artifact in this release 84.2% of rows have negative "days_since_update". Staleness is retained only as a reason-code booster.
- **Not a security tool.** Nothing in this work detects malicious, deceptive, or abusive content. Unusual combinations are surfaced for human review, not as alarms.

In [24]:
print(f"Sync-artifact rows: {(df['days_since_update'] < 0).sum():,} "
      f"({(df['days_since_update'] < 0).mean():.1%})")
print(f"Pages with no future-window comparison dropped: "
      f"{((df['imp_apr_may'].isna()) | (df['impressions_30d'] == 0)).sum():,}")
print()
print("Claim discipline: observed / measured / directional / decision-support.")
print("No causal claim, no Google-algorithm claim, no malicious-content claim.")

Sync-artifact rows: 148,782 (84.2%)
Pages with no future-window comparison dropped: 0

Claim discipline: observed / measured / directional / decision-support.
No causal claim, no Google-algorithm claim, no malicious-content claim.


## 6. Ranked recommendations

The action playbook, ordered by what a content team should do first.

1. Rewrite title/meta on the top of the 'review_ctr' queue. 1,368 pages flagged. The top row alone has 80,821 impressions at 0.04% CTR, ranking 1.5 a large, cheap-to-test upside.

2. Add a minimum-impressions floor to any future model. LR's top-50 ranks were dominated by 3-impression pages; the baseline's 'visible ≥ 500' floor is what makes it win at P@50.

3. Do not deploy the LR as the primary ranker. It loses to the hand rule on the same split at the metric that matters (P@50).

4. Re-audit the CTR-vs-tier interaction explicitly. The rule captures it; LR sees only raw CTR and misses it. A future model could add 'ctr_rel_tier' as a feature.

5. Hold off on staleness-based filtering until 'content_updated_date' is fixed upstream.

6. Treat the queue as decision-support. No automated edits, no auto-publish. Human review is in the loop.

In [25]:
print("Top 10 of the queue (as of the capstone run):")
print(df.sort_values("baseline_score", ascending=False)
        .head(10)[["content_hash_id","baseline_score","impressions_30d","ctr_30d","avg_position_30d"]]
        .to_string(index=False))

Top 10 of the queue (as of the capstone run):
         content_hash_id  baseline_score  impressions_30d  ctr_30d  avg_position_30d
content_306bc78dff1eb683         80821.0          80821.0 0.043306          1.488604
content_c46df0fa61530d86         70398.0          70398.0 0.059661          1.556258
content_b2b85c287474668d         65304.0          65304.0 0.093409          1.541702
content_fc67675904376267         60172.0          60172.0 0.029914          2.261303
content_ff8941941141101f         44217.0          44217.0 0.090463          2.397902
content_7f52754cb72a5991         43135.0          43135.0 0.064912          2.502348
content_d61fc394d10cba41         38000.0          38000.0 0.002632          2.740744
content_a07d1e3236680189         35083.0          35083.0 0.059858          2.838985
content_eea13d1934ec6e1a         34448.0          34448.0 0.075476          2.893233
content_805fd45594ea2398         30792.0          30792.0 0.038971          2.920553


## 7. Artifacts the paper embeds


- (docs/figures/action_mix.png) action-label distribution.
- (docs/figures/score_by_rank.png) baseline score by rank (top 5,000).
- (work/outputs/playbook_metrics.json) standing metrics (P@20, P@50, base rate, reason/action counts).
- (work/outputs/baseline_action_score.csv) the full ranked queue (regenerated on every run; excluded from git by the leak-guard).

In [26]:
# Regenerating both figures and write them to docs/figures/
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7,4))
df["action_label"].value_counts().plot(kind="bar", ax=ax, color="#2b6cb0")
ax.set_title("Action-label distribution — March 2026 prior window")
ax.set_ylabel("pages"); ax.set_xlabel("")
plt.tight_layout(); fig.savefig("docs/figures/action_mix.png", dpi=120); plt.close(fig)

fig, ax = plt.subplots(figsize=(7,4))
df.sort_values("baseline_score", ascending=False)["baseline_score"].head(5000).plot(
    ax=ax, color="#c53030")
ax.set_title("Baseline score by rank (top 5,000 rows)")
ax.set_ylabel("baseline_score"); ax.set_xlabel("rank")
plt.tight_layout(); fig.savefig("docs/figures/score_by_rank.png", dpi=120); plt.close(fig)

print("Wrote docs/figures/action_mix.png and docs/figures/score_by_rank.png")

Wrote docs/figures/action_mix.png and docs/figures/score_by_rank.png


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
